In [1]:
import logging

import pandas as pd

import data.breathe_data as bd
import data.helpers as dh
import data.sanity_checks as sanity_checks
import models.var_builders as var_builders


Exploring the CF Trust registry to evaluate if model output (synthesizing FEV1, FEF25-75 on 2 days) can be used to improve ML achieved from each indidivually 

In [2]:
# Vars
year = 2022
sheet_name=f"{year} Annual Review"

In [3]:
cols2read = [
    "s01caseid_original",
    # "s01sex",
    "s01height",
    "s01encounterageyears",
    "s03cliqtrfev1",  # Value at annual review
    # "s03clibestfev1",
    "s03clifef2575",  # Value at annual review
]

df_19 = pd.read_excel(
    dh.get_path_to_main() + f"ExcelFiles/CFT/CF_Registry_2019_23_Floto_Output.xlsx",
    sheet_name=sheet_name,
    usecols=cols2read,
)

KeyboardInterrupt: 

In [ ]:
cols2read = ["s01caseid_original", "s01sex"]

df_demo = pd.read_excel(
    dh.get_path_to_main() + f"ExcelFiles/CFT/CF_Registry_2019_23_Floto_Output.xlsx",
    sheet_name="Demographics",
    usecols=cols2read,
)

In [ ]:
print(f"Loaded {df_19.shape[0]} entries")
df_19_1 = df_19.dropna()
print(
    f"{df_19_1.shape[0]} after removing all NaN (note: sex, age, height cols were complete)"
)
df_19_2 = df_19_1.merge(df_demo, on="s01caseid_original")

# Data cleaning and formatting
df_19_2 = df_19_2.rename(columns={
    "s01caseid_original": "ID",
    "s01encounterageyears": "Age",
    "s01sex": "Sex",
    "s01height": "Height",
    "s03cliqtrfev1": "FEV1",
    "s03clifef2575": "FEF2575"
})

sanity_checks.data_types(df_19_2)

df_19_2.Sex = df_19_2.Sex.apply(lambda row: "Female" if "F" else "Male")
df_19_2.Height = df_19_2.Height.round()
df_19_2["Date Recorded"] = f"{year}-01-01"
df_19_2["Date Recorded"] = pd.to_datetime(df_19_2["Date Recorded"]).dt.date

Loaded 10251 entries
4315 after removing all NaN (note: sex, age, height cols were complete)


In [80]:
df_19_2.to_excel(dh.get_path_to_main() + "ExcelFiles/CF_Registry_processed.xlsx", index=False)

In [4]:
df = bd.load_meas_from_excel("CF_Registry_processed", study_folder="CFT")

INFO:root:* Checking for same day measurements *


In [11]:
df['ecFEV1'] = df['FEV1']
df['ecFEF2575'] = df['FEF2575']
df['ecFEF2575%ecFEV1'] = df['FEF2575']/df.FEV1 * 100

In [12]:
# Keep adults
print(f"{(df.Age >= 18).sum()} entries after removing <18yr")
df = df[df.Age >= 18]

df = bd.calc_predicted_FEV1_LMS_df(df)
df = bd.calc_FEV1_prct_predicted_df(df)

2046 entries after removing <18yr


In [13]:
# Add indices for model
height = df.Height.iloc[0]
age = df.Age.iloc[0]
sex = df.Sex.iloc[0]
ar_prior="uniform"
ecfev1_noise_model_cpt_suffix = "_std_add_mult_ecfev1"
ar_fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"
(
    HFEV1,
    uFEV1,
    ecFEV1,
    AR,
    ecFEF2575prctecFEV1,
) = var_builders.fev1_fef2575_point_in_time_model_noise_shared_healthy_vars(
    height,
    age,
    sex,
    ar_prior,
    ecfev1_noise_model_cpt_suffix,
    ar_fef2575_cpt_suffix,
)

df[f"idx {ecFEV1.name}"] = df.apply(
    lambda row: ecFEV1.get_bin_idx_for_value(row["ecFEV1"]), axis=1
)
df[f"idx {ecFEF2575prctecFEV1.name}"] = df.apply(
    lambda row: ecFEF2575prctecFEV1.get_bin_idx_for_value(row["ecFEF2575%ecFEV1"]),
    axis=1,
)

In [15]:
df.to_excel(dh.get_path_to_main() + "ExcelFiles/CFT/CF_Registry_processed_with_idx.xlsx", index=False)